In [1]:
import argparse
import inspect
import os
import time
from types import SimpleNamespace

import matplotlib.pyplot as plt


import jax
from jax import jit, lax, random
from jax.example_libraries import stax
import jax.numpy as jnp
from jax.random import PRNGKey

import numpyro
from numpyro import optim
import numpyro.distributions as dist
from numpyro.examples.datasets import MNIST, load_dataset
from numpyro.infer import SVI, Trace_ELBO
from CustomModules.normalizing_flow import normalizing_flow

RESULTS_DIR = "./vae_example_results"
os.makedirs(RESULTS_DIR, exist_ok=True)



/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


def model(batch, **kwargs):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)
    # Configue Flow
    flow_transform = numpyro.module(
        "flow", 
        normalizing_flow(
            input_dim=out_dim, 
            hidden_dims=[out_dim, out_dim],
            steps=5,
            inv = False
        ),
        input_shape=(batch_dim, out_dim)
    )


    with numpyro.plate("batch", batch_dim):
        z = numpyro.sample("z", dist.Normal(0, 1).expand([out_dim]).to_event(1))
        base_dist = dist.Delta(v=z)
        flow_dist = dist.TransformedDistribution(base_dist, flow_transform())


        img_loc = numpyro.sample("img_loc", flow_dist)
        return numpyro.sample("obs", dist.Bernoulli(img_loc).to_event(1), obs=batch)


def encoder(hidden_dim, z_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.FanOut(2),
        stax.parallel(
            stax.Dense(z_dim, W_init=stax.randn()),
            stax.serial(stax.Dense(z_dim, W_init=stax.randn()), stax.Exp),
        ),
    )

def guide(batch, hidden_dim=400):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)
    encode = numpyro.module("encoder", encoder(hidden_dim, out_dim), (batch_dim, out_dim))
    z_loc, z_std = encode(batch)
    with numpyro.plate("batch", batch_dim):
        return numpyro.sample("z", dist.Normal(z_loc, z_std).to_event(1))



@jit
def binarize(rng_key, batch):
    return random.bernoulli(rng_key, batch).astype(batch.dtype)


def main(args):
    encoder_nn = encoder(args.hidden_dim, 784)
    flow = normalizing_flow(
            input_dim=784, 
            hidden_dims=[784],
            steps=5,
            inv = False
        )
    
    adam = optim.Adam(args.learning_rate)
    svi = SVI(
        model, guide, adam, Trace_ELBO(), hidden_dim=args.hidden_dim
    )
    rng_key = PRNGKey(0)
    train_init, train_fetch = load_dataset(
        MNIST, batch_size=args.batch_size, split="train"
    )
    test_init, test_fetch = load_dataset(
        MNIST, batch_size=args.batch_size, split="test"
    )
    num_train, train_idx = train_init()
    print("num_train: ", num_train)
    rng_key, rng_key_binarize, rng_key_init = random.split(rng_key, 3)
    sample_batch = binarize(rng_key_binarize, train_fetch(0, train_idx)[0])
    svi_state = svi.init(rng_key_init, sample_batch)

    @jit
    def epoch_train(svi_state, rng_key, train_idx):
        def body_fn(i, val):
            jax.debug.print("Current iteration: {i}", i=i)
            loss_sum, svi_state = val
            rng_key_binarize = random.fold_in(rng_key, i)
            batch = binarize(rng_key_binarize, train_fetch(i, train_idx)[0])
            svi_state, loss = svi.update(svi_state, batch)
            loss_sum += loss
            return loss_sum, svi_state

        return lax.fori_loop(0, num_train, body_fn, (0.0, svi_state))

    @jit
    def eval_test(svi_state, rng_key, test_idx):
        def body_fun(i, loss_sum):
            rng_key_binarize = random.fold_in(rng_key, i)
            batch = binarize(rng_key_binarize, test_fetch(i, test_idx)[0])
            # FIXME: does this lead to a requirement for an rng_key arg in svi_eval?
            loss = svi.evaluate(svi_state, batch) / len(batch)
            loss_sum += loss
            return loss_sum

        loss = lax.fori_loop(0, num_test, body_fun, 0.0)
        loss = loss / num_test
        return loss

    def reconstruct_img(epoch, rng_key):
        img = test_fetch(0, test_idx)[0][0]
        plt.imsave(
            os.path.join(RESULTS_DIR, "original_epoch={}.png".format(epoch)),
            img,
            cmap="gray",
        )
        rng_key_binarize, rng_key_sample = random.split(rng_key)
        test_sample = binarize(rng_key_binarize, img)
        params = svi.get_params(svi_state)
        z_mean, z_var = encoder_nn[1](
            params["encoder$params"], test_sample.reshape([1, -1])
        )
        z = dist.Normal(z_mean, z_var).sample(rng_key_sample)
        flow_transform = flow[1](params["flow$params"])

        img_loc = flow_transform()(z)
        plt.imsave(
            os.path.join(RESULTS_DIR, "recons_epoch={}.png".format(epoch)),
            img_loc,
            cmap="gray",
        )
    
    for i in range(args.num_epochs):
        print("Epoch, ", i, "begin")
        rng_key, rng_key_train, rng_key_test, rng_key_reconstruct = random.split(
            rng_key, 4
        )
        t_start = time.time()
        num_train, train_idx = train_init()
        _, svi_state = epoch_train(svi_state, rng_key_train, train_idx)
        rng_key, rng_key_test, rng_key_reconstruct = random.split(rng_key, 3)
        num_test, test_idx = test_init()
        test_loss = eval_test(svi_state, rng_key_test, test_idx)
        reconstruct_img(i, rng_key_reconstruct)

        rng_key, rng_key_mse = jax.random.split(rng_key)

        print(
            "Epoch {}: loss = {} ({:.2f} s.)".format(
                i, test_loss, time.time() - t_start
            )
        )

def main_with_args(num_epochs=15, lr=5.0e-3, batch_size=128, z_dim=50, hidden_dim=400):

    args = SimpleNamespace()
    args.num_epochs = num_epochs
    args.learning_rate =lr
    args.batch_size = batch_size
    args.z_dim = z_dim
    args.hidden_dim = hidden_dim

    return main(args)


main_with_args(z_dim=784, batch_size=50)

num_train:  1200
0 : (50, 784)
1 : (50, 784)
2 : (50, 784)
3 : (50, 784)
4 : (50, 784)
Epoch,  0 begin


/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/jax/_src/api_util.py:293: UserWarning: Found vars in model but not guide: {'img_loc'}
  return _fun(*args, **kwargs)


Current iteration: 0
Current iteration: 1
Current iteration: 2
Current iteration: 3
Current iteration: 4
Current iteration: 5
Current iteration: 6
Current iteration: 7
Current iteration: 8
Current iteration: 9
Current iteration: 10
Current iteration: 11
Current iteration: 12
Current iteration: 13
Current iteration: 14
Current iteration: 15
Current iteration: 16
Current iteration: 17
Current iteration: 18
Current iteration: 19
Current iteration: 20
Current iteration: 21
Current iteration: 22
Current iteration: 23
Current iteration: 24
Current iteration: 25
Current iteration: 26
Current iteration: 27
Current iteration: 28
Current iteration: 29
Current iteration: 30
Current iteration: 31
Current iteration: 32
Current iteration: 33
Current iteration: 34
Current iteration: 35
Current iteration: 36
Current iteration: 37
Current iteration: 38
Current iteration: 39
Current iteration: 40
Current iteration: 41
Current iteration: 42
Current iteration: 43
Current iteration: 44
Current iteration: 4

E0301 15:16:16.334017   13101 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/asyncio/base_events.py", line 1987, in _ru

JaxRuntimeError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once
  File "/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/asyncio/event